# Biểu diễn 5 câu bằng vector TF-IDF

Notebook này dành cho người mới bắt đầu. Ta sẽ đi từ khái niệm cơ bản, nhập **5 câu bất kỳ**, biến mỗi câu thành một vector số bằng TF-IDF, rồi đọc và giải thích kết quả.

## Mục tiêu

Sau khi chạy xong notebook, bạn sẽ hiểu:

1. Vì sao máy tính cần biến câu chữ thành số.
2. TF, IDF và TF-IDF có ý nghĩa gì.
3. Cách dùng `TfidfVectorizer` của scikit-learn.
4. Cách đọc ma trận gồm 5 vector TF-IDF.


## 1. Ý tưởng cơ bản

Máy tính không hiểu trực tiếp ý nghĩa của câu như con người. Nhiều thuật toán chỉ làm việc với số, nên ta cần biến mỗi câu thành một dãy số gọi là **vector**.

Ví dụ, nếu toàn bộ 5 câu chỉ có ba từ khác nhau là `mèo`, `chó`, `nhà`, ta quy ước thứ tự cột là:

`[mèo, chó, nhà]`

Mỗi câu sẽ được biểu diễn bởi một vector có 3 số. Số tại mỗi vị trí cho biết từ tương ứng quan trọng đến mức nào trong câu. TF-IDF giúp ta tính các con số đó.

> Trong bài này: một câu được xem là một **document (văn bản)**; một từ được gọi là một **term**; danh sách tất cả từ khác nhau được gọi là **vocabulary (bộ từ vựng)**.


## 2. TF-IDF là gì?

**TF-IDF = Term Frequency × Inverse Document Frequency**.

### TF — từ xuất hiện nhiều thế nào trong một câu?

Một từ xuất hiện nhiều lần trong câu thường liên quan đến nội dung câu đó. Hiểu đơn giản, TF tăng khi số lần xuất hiện của từ trong câu tăng.

### IDF — từ đó hiếm thế nào trong cả 5 câu?

Nếu một từ có mặt trong hầu hết mọi câu, từ đó ít giúp phân biệt các câu. Ngược lại, từ chỉ xuất hiện trong ít câu sẽ có IDF cao hơn.

Scikit-learn dùng công thức IDF đã làm trơn mặc định:

$$idf(t) = \log\left(\frac{1+n}{1+df(t)}\right)+1$$

Trong đó:

- $n$ là tổng số câu, ở đây $n=5$.
- $df(t)$ là số câu có chứa từ $t$.
- Cộng 1 giúp tránh trường hợp chia cho 0.

Sau đó, trọng số ban đầu của từ là:

$$tfidf(t,d) = tf(t,d) \times idf(t)$$

Cuối cùng, scikit-learn chuẩn hóa mỗi vector theo chuẩn L2 để độ dài vector bằng 1. Vì vậy giá trị hiển thị không chỉ là phép nhân thô ở trên. Một giá trị TF-IDF càng lớn nghĩa là từ đó càng nổi bật trong câu tương ứng.


In [1]:
# Thư viện dùng để tạo bảng cho kết quả dễ đọc
import pandas as pd

# Công cụ chuyển văn bản thành vector TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

# Công cụ tính độ giống nhau giữa các vector (phần mở rộng ở cuối)
from sklearn.metrics.pairwise import cosine_similarity

# Hiển thị số thực gọn hơn trong bảng
pd.options.display.float_format = "{:.3f}".format


## 3. Chuẩn bị đúng 5 câu

Có hai cách chạy:

- Giữ `TU_NHAP = False`: notebook dùng 5 câu mẫu và có thể chạy toàn bộ ngay.
- Đổi thành `TU_NHAP = True`: chương trình lần lượt yêu cầu bạn gõ 5 câu bất kỳ.

Các câu nên có một vài từ chung và một vài từ riêng để kết quả dễ quan sát.


In [2]:
# Đổi False thành True nếu bạn muốn tự nhập 5 câu từ bàn phím.
TU_NHAP = False

if TU_NHAP:
    cau = []
    for i in range(1, 6):
        # strip() bỏ khoảng trắng thừa ở đầu và cuối câu.
        noi_dung = input(f"Nhập câu thứ {i}: " ).strip()
        cau.append(noi_dung)
else:
    # 5 câu mẫu. Bạn cũng có thể sửa trực tiếp nội dung tại đây.
    cau = [
        "Tôi thích học trí tuệ nhân tạo",
        "Trí tuệ nhân tạo giúp máy tính học từ dữ liệu",
        "Dữ liệu rất quan trọng trong học máy",
        "Tôi đang học lập trình Python",
        "Python được dùng nhiều trong trí tuệ nhân tạo",
    ]

# Kiểm tra đầu vào để báo lỗi rõ ràng thay vì lỗi khó hiểu ở bước sau.
if len(cau) != 5:
    raise ValueError("Cần có đúng 5 câu.")
if any(not mot_cau for mot_cau in cau):
    raise ValueError("Mỗi câu phải có ít nhất một ký tự, không được để trống.")

print("5 câu sẽ được xử lý:")
for i, mot_cau in enumerate(cau, start=1):
    print(f"Câu {i}: {mot_cau}")


5 câu sẽ được xử lý:
Câu 1: Tôi thích học trí tuệ nhân tạo
Câu 2: Trí tuệ nhân tạo giúp máy tính học từ dữ liệu
Câu 3: Dữ liệu rất quan trọng trong học máy
Câu 4: Tôi đang học lập trình Python
Câu 5: Python được dùng nhiều trong trí tuệ nhân tạo


## 4. Tạo vector TF-IDF

`TfidfVectorizer` thực hiện các bước sau:

1. Chuyển chữ hoa thành chữ thường vì `lowercase=True`.
2. Tách câu thành các từ (token). Dấu câu không trở thành một cột.
3. Tạo bộ từ vựng từ tất cả từ khác nhau trong 5 câu.
4. Tính TF-IDF cho từng từ trong từng câu.
5. Chuẩn hóa mỗi vector theo chuẩn L2 vì `norm="l2"`.

> Lưu ý: cách tách từ mặc định phù hợp với ví dụ cơ bản. Với tiếng Việt, cụm như `trí tuệ nhân tạo` sẽ được xem là bốn từ riêng. Các hệ thống tiếng Việt nâng cao thường có thêm bước tách từ chuyên dụng.


In [3]:
# Khởi tạo bộ biến đổi. Các tham số được ghi rõ để người mới dễ theo dõi.
vectorizer = TfidfVectorizer(
    lowercase=True,    # Xem "Python" và "python" là cùng một từ
    norm="l2",        # Chuẩn hóa độ dài mỗi vector thành 1
    use_idf=True,      # Có sử dụng thành phần IDF
    smooth_idf=True,   # Dùng công thức IDF có làm trơn
)

# fit_transform vừa học bộ từ vựng, vừa biến 5 câu thành ma trận TF-IDF.
ma_tran_tfidf = vectorizer.fit_transform(cau)

print("Kiểu dữ liệu ban đầu:", type(ma_tran_tfidf))
print("Kích thước ma trận (số câu, số từ khác nhau):", ma_tran_tfidf.shape)


Kiểu dữ liệu ban đầu: <class 'scipy.sparse._csr.csr_matrix'>
Kích thước ma trận (số câu, số từ khác nhau): (5, 24)


### Vì sao kết quả ban đầu là sparse matrix?

Trong dữ liệu văn bản thật, bộ từ vựng có thể có hàng chục nghìn từ, nhưng mỗi câu chỉ chứa một số ít từ. Phần lớn phần tử của vector bằng 0. **Sparse matrix (ma trận thưa)** chỉ lưu các giá trị khác 0 để tiết kiệm bộ nhớ. Với đúng 5 câu nhỏ, ta chuyển nó thành bảng đặc để dễ quan sát.


In [4]:
# Mỗi tên từ tương ứng với đúng một cột của vector.
tu_vung = vectorizer.get_feature_names_out()

print(f"Bộ từ vựng có {len(tu_vung)} từ:")
print(tu_vung)

print("\nThứ tự vị trí trong vector (bắt đầu từ 0):")
for vi_tri, tu in enumerate(tu_vung):
    print(f"Vị trí {vi_tri:2d}: {tu}")


Bộ từ vựng có 24 từ:
['dùng' 'dữ' 'giúp' 'học' 'liệu' 'lập' 'máy' 'nhiều' 'nhân' 'python'
 'quan' 'rất' 'thích' 'trong' 'trình' 'trí' 'trọng' 'tuệ' 'tính' 'tôi'
 'tạo' 'từ' 'đang' 'được']

Thứ tự vị trí trong vector (bắt đầu từ 0):
Vị trí  0: dùng
Vị trí  1: dữ
Vị trí  2: giúp
Vị trí  3: học
Vị trí  4: liệu
Vị trí  5: lập
Vị trí  6: máy
Vị trí  7: nhiều
Vị trí  8: nhân
Vị trí  9: python
Vị trí 10: quan
Vị trí 11: rất
Vị trí 12: thích
Vị trí 13: trong
Vị trí 14: trình
Vị trí 15: trí
Vị trí 16: trọng
Vị trí 17: tuệ
Vị trí 18: tính
Vị trí 19: tôi
Vị trí 20: tạo
Vị trí 21: từ
Vị trí 22: đang
Vị trí 23: được


In [5]:
# toarray() chuyển ma trận thưa thành mảng số thông thường để tạo DataFrame.
bang_tfidf = pd.DataFrame(
    ma_tran_tfidf.toarray(),
    columns=tu_vung,
    index=[f"Câu {i}" for i in range(1, 6)],
)

print("Ma trận TF-IDF (5 hàng chính là 5 vector):")
bang_tfidf


Ma trận TF-IDF (5 hàng chính là 5 vector):


,dùng,dữ,giúp,học,liệu,lập,máy,nhiều,nhân,python,...,trình,trí,trọng,tuệ,tính,tôi,tạo,từ,đang,được
Câu 1,0.000,0.000,0.000,0.290,0.000,0.000,0.000,0.000,0.345,0.000,...,0.000,0.345,0.000,0.345,0.000,0.416,0.345,0.000,0.000,0.000
Câu 2,0.000,0.304,0.376,0.212,0.304,0.000,0.304,0.000,0.252,0.000,...,0.000,0.252,0.000,0.252,0.376,0.000,0.252,0.376,0.000,0.000
Câu 3,0.000,0.332,0.000,0.232,0.332,0.000,0.332,0.000,0.000,0.000,...,0.000,0.000,0.411,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Câu 4,0.000,0.000,0.000,0.262,0.000,0.465,0.000,0.000,0.000,0.375,...,0.465,0.000,0.000,0.000,0.000,0.375,0.000,0.000,0.465,0.000
Câu 5,0.405,0.000,0.000,0.000,0.000,0.000,0.000,0.405,0.271,0.327,...,0.000,0.271,0.000,0.271,0.000,0.000,0.271,0.000,0.000,0.405


## 5. Cách đọc ma trận

- **Mỗi hàng** là vector TF-IDF của một câu. Vì có 5 câu nên có 5 hàng.
- **Mỗi cột** ứng với một từ trong bộ từ vựng.
- Giá trị **0** nghĩa là từ đó không xuất hiện trong câu.
- Giá trị **lớn hơn 0** cho biết mức độ nổi bật của từ trong câu. Giá trị lớn hơn thường có nghĩa là từ xuất hiện nhiều trong câu và/hoặc hiếm trong toàn bộ 5 câu.
- Không nên so sánh máy móc giá trị giữa hai bộ 5 câu khác nhau vì bộ từ vựng và IDF sẽ thay đổi.

Bảng tiếp theo chỉ in các từ có trọng số khác 0 của từng câu, nên dễ đọc hơn ma trận rộng.


In [6]:
for i, mot_cau in enumerate(cau):
    # Lấy hàng thứ i, rồi chỉ giữ các cột có trọng số lớn hơn 0.
    trong_so = bang_tfidf.iloc[i]
    tu_quan_trong = trong_so[trong_so > 0].sort_values(ascending=False)

    print(f"\nCâu {i + 1}: {mot_cau}")
    for tu, diem in tu_quan_trong.items():
        print(f"  {tu:<15} -> {diem:.3f}")



Câu 1: Tôi thích học trí tuệ nhân tạo
  thích           -> 0.516
  tôi             -> 0.416
  nhân            -> 0.345
  tuệ             -> 0.345
  trí             -> 0.345
  tạo             -> 0.345
  học             -> 0.290

Câu 2: Trí tuệ nhân tạo giúp máy tính học từ dữ liệu
  giúp            -> 0.376
  tính            -> 0.376
  từ              -> 0.376
  liệu            -> 0.304
  dữ              -> 0.304
  máy             -> 0.304
  trí             -> 0.252
  tuệ             -> 0.252
  nhân            -> 0.252
  tạo             -> 0.252
  học             -> 0.212

Câu 3: Dữ liệu rất quan trọng trong học máy
  rất             -> 0.411
  trọng           -> 0.411
  quan            -> 0.411
  dữ              -> 0.332
  liệu            -> 0.332
  máy             -> 0.332
  trong           -> 0.332
  học             -> 0.232

Câu 4: Tôi đang học lập trình Python
  lập             -> 0.465
  trình           -> 0.465
  đang            -> 0.465
  python          -> 0.375
  tôi         

## 6. Xem riêng IDF để hiểu từ phổ biến và từ hiếm

IDF không phụ thuộc vào riêng một câu mà phụ thuộc vào việc từ xuất hiện trong bao nhiêu câu. Trong cùng bộ 5 câu:

- IDF thấp hơn: từ có mặt trong nhiều câu hơn.
- IDF cao hơn: từ hiếm hơn, có mặt trong ít câu hơn.


In [7]:
bang_idf = pd.DataFrame({
    "Từ": tu_vung,
    "IDF": vectorizer.idf_,
}).sort_values(["IDF", "Từ"])

bang_idf.reset_index(drop=True)


,Từ,IDF
0,học,1.182
1,nhân,1.405
2,trí,1.405
3,tuệ,1.405
4,tạo,1.405
5,dữ,1.693
6,liệu,1.693
7,máy,1.693
8,python,1.693
9,trong,1.693


## 7. Kiểm tra độ dài vector

Do ta dùng chuẩn hóa L2, căn bậc hai của tổng bình phương các phần tử trong mỗi vector sẽ xấp xỉ 1:

$$\|v\|_2 = \sqrt{v_1^2 + v_2^2 + \cdots + v_m^2} \approx 1$$


In [8]:
# Tính chuẩn L2 mà không cần thêm thư viện: bình phương, cộng, rồi căn bậc hai.
do_dai_vector = (bang_tfidf.pow(2).sum(axis=1)) ** 0.5
do_dai_vector.rename("Độ dài L2").to_frame()


,Độ dài L2
Câu 1,1.000
Câu 2,1.000
Câu 3,1.000
Câu 4,1.000
Câu 5,1.000


## 8. Phần mở rộng: câu nào giống nhau nhất?

Khi đã có vector, ta có thể đo độ giống nhau giữa hai câu bằng **cosine similarity**. Giá trị thường nằm từ 0 đến 1 đối với các vector TF-IDF này:

- Gần 1: hai câu có nhiều từ quan trọng chung.
- Gần 0: hai câu có ít hoặc không có từ chung.

Đây chỉ là phép so sánh dựa trên từ xuất hiện, chưa hiểu sâu ngữ nghĩa như con người.


In [9]:
ma_tran_tuong_dong = cosine_similarity(ma_tran_tfidf)
bang_tuong_dong = pd.DataFrame(
    ma_tran_tuong_dong,
    index=[f"Câu {i}" for i in range(1, 6)],
    columns=[f"Câu {i}" for i in range(1, 6)],
)

bang_tuong_dong


,Câu 1,Câu 2,Câu 3,Câu 4,Câu 5
Câu 1,1.000,0.410,0.067,0.232,0.375
Câu 2,0.410,1.000,0.351,0.056,0.273
Câu 3,0.067,0.351,1.000,0.061,0.108
Câu 4,0.232,0.056,0.061,1.000,0.123
Câu 5,0.375,0.273,0.108,0.123,1.000


In [10]:
# Tìm cặp câu khác nhau có độ tương đồng lớn nhất.
cap_tot_nhat = None
diem_tot_nhat = -1.0

for i in range(5):
    for j in range(i + 1, 5):  # j bắt đầu sau i để không so một câu với chính nó
        diem = ma_tran_tuong_dong[i, j]
        if diem > diem_tot_nhat:
            diem_tot_nhat = diem
            cap_tot_nhat = (i, j)

i, j = cap_tot_nhat
print(f"Cặp giống nhau nhất: Câu {i + 1} và Câu {j + 1}")
print(f"Độ tương đồng cosine: {diem_tot_nhat:.3f}")
print("-", cau[i])
print("-", cau[j])


Cặp giống nhau nhất: Câu 1 và Câu 2
Độ tương đồng cosine: 0.410
- Tôi thích học trí tuệ nhân tạo
- Trí tuệ nhân tạo giúp máy tính học từ dữ liệu


## 9. Kết luận

Quy trình hoàn chỉnh của bài là:

**5 câu → tách từ → tạo bộ từ vựng → tính TF và IDF → chuẩn hóa → 5 vector số**

Điểm cần nhớ:

1. Số chiều của mỗi vector bằng số từ khác nhau trong bộ từ vựng.
2. Cả 5 vector luôn có cùng số chiều và cùng thứ tự cột.
3. Từ không có trong một câu nhận trọng số 0 ở câu đó.
4. Từ nổi bật trong một câu nhưng ít phổ biến trong toàn bộ tập thường nhận trọng số cao.
5. TF-IDF dựa trên từ và tần suất; nó không thật sự hiểu ngữ cảnh hay ý nghĩa sâu của câu.

### Gợi ý tự thực hành

- Đổi `TU_NHAP = True` rồi nhập 5 câu của riêng bạn.
- Thử cho một từ xuất hiện trong cả 5 câu và xem IDF của nó thay đổi thế nào.
- Lặp lại một từ nhiều lần trong một câu và quan sát TF-IDF.
- Thử hai câu khác nghĩa nhưng dùng nhiều từ giống nhau để thấy giới hạn của TF-IDF.
